In [3]:
import pandas as pd
import glob
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn import datasets, linear_model
from sklearn.model_selection import cross_val_score


In [4]:
###
# All variables releated data are located here to make an order
###
PATH="C:\\Users\\emirc\\Desktop\\AAR\\data\\csv\\datas/*" #Refer the path having the csv files
#ANIMALS_USED=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18]   #Animal numbers to study on 
ANIMALS_USED=[1,2]
COLUMNS_REMOVE=['Mx','My','Mz','Gx','Gy','Gz','G3D','M3D'] #Columns to delete such as Gx Gy Gz
ACTIVITIES_MERGE={'running-natural': 'running',
                 'running-rider': 'running',
                  'trotting-natural': 'trotting',    #Having nearly the same meaning
                  'trotting-rider': 'trotting',} 
ACTIVITIESTOWINDOW=['standing','standing-rider','walking-natural','walking-rider','trotting','running','galloping-natural','galloping-rider','jumping','grazing','eating','shaking','scratch-biting','rubbing','fighting','rolling','scared','head-shake']
TIME_PERIODS = 200
STEP_DISTANCE = 100
filePaths = sorted(glob.glob(PATH))


In [5]:
def createDataFrame(filePaths):
    dataFrame=pd.DataFrame()
    for fileName in filePaths:
        csv=pd.read_csv(fileName,low_memory=False)
        dataFrame=dataFrame.append(csv)
    
    #Removing columns
    dataFrame.drop(COLUMNS_REMOVE,axis=1,inplace=True)
    le = preprocessing.LabelEncoder()
    dataFrame['ActivityEncoded'] = le.fit_transform(dataFrame['label'].values.ravel())

    
    #Removing needless horses
    dataFrame=dataFrame[dataFrame['subject'].isin(ANIMALS_USED)]
    
    #Merging activities
    for key in ACTIVITIES_MERGE:
        dataFrame['label'] = dataFrame['label'].replace(to_replace=key, value=ACTIVITIES_MERGE.get(key))
       
    #deleting unlabeled data
    dataFrame=dataFrame[~(dataFrame['label'].isin(['unknown']))]

    #removing nulls
    dataFrame=dataFrame.dropna(axis=0, how='any')


    return dataFrame

In [6]:
def lowBufferFiltering(dataFrame):
    sos = signal.butter(N=3, Wn=30, btype='lowpass', fs=100, output='sos')
    dataFrame['Ax'] = signal.sosfilt(sos, dataFrame['Ax'])
    dataFrame['Ay'] = signal.sosfilt(sos, dataFrame['Ay'])
    dataFrame['Az'] = signal.sosfilt(sos, dataFrame['Az'])
    return dataFrame
    

In [7]:
def windowing(dataFrame,labelName):
    windows = []
    labels = []
    for i in range(0, len(dataFrame)-TIME_PERIODS, STEP_DISTANCE):
        axs = dataFrame['Ax'].values[i: i + TIME_PERIODS]
        ays = dataFrame['Ay'].values[i: i + TIME_PERIODS]
        azs = dataFrame['Az'].values[i: i + TIME_PERIODS]
    
        # Retrieve the most often used label in this segment
        label = stats.mode(dataFrame[labelName][i: i + TIME_PERIODS])[0][0]
        windows.append([axs, ays, azs])
        labels.append(label)
        
    reshaped_windows = np.asarray(windows, dtype= np.float32).reshape(-1, TIME_PERIODS, 3)
    labels = np.asarray(labels)
    return reshaped_windows, labels

In [8]:
def windowActivities(dataFrame):
    windowedDatas=[]
    for activity in ACTIVITIESTOWINDOW:
        try:
            selectedDataFrame=dataFrame.loc[dataFrame['label']==activity]
        except KeyError:
            continue
        window, labels_=windowing(selectedDataFrame,LABEL)
        num_time_periods, num_sensors = window.shape[1], window.shape[2]
        input_shape = (num_time_periods*num_sensors)
        window = window.reshape(window.shape[0], input_shape)
        selectedDataFrame=pd.DataFrame(window)
        windowedDatas.append(selectedDataFrame)
    return windowedDatas

In [9]:
def featureScaling(dataFrame):
    mean=dataFrame.mean()
    print(mean)
    
    

In [11]:
def kFCrossValidation(dataSet):
    #below lines will be fixed
    x = dataFrame.iloc[:,2:1941].values
    y = dataFrame.iloc[:,0].values
    cv = KFold(n_splits=10, random_state=1, shuffle=True)
    model = RandomForestClassifier(random_state=1)
    scores = cross_val_score(model, x, y, scoring='accuracy', cv=cv, n_jobs=-1)
    print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
    

In [12]:
dataFrame=createDataFrame(filePaths)

In [13]:
print(dataFrame.columns)
LABEL = 'ActivityEncoded'
le = preprocessing.LabelEncoder()
dataFrame[LABEL] = le.fit_transform(dataFrame['label'].values.ravel())


Index(['Ax', 'Ay', 'Az', 'A3D', 'label', 'segment', 'subject',
       'ActivityEncoded'],
      dtype='object')


In [14]:
#filtering
dataFrame=lowBufferFiltering(dataFrame)


In [15]:
print(dataFrame)
windowedResults=windowActivities(dataFrame)

                Ax        Ay        Az        A3D            label  segment  \
777585    2.242763 -0.607748  0.781215   9.541875    walking-rider    25848   
777586    8.171002 -1.761539  2.803087  11.284639    walking-rider    25848   
777587   12.236442 -1.560116  4.644168  12.953753    walking-rider    25848   
777588   11.434799 -0.382914  5.664330  12.188769    walking-rider    25848   
777589   10.231657 -0.558354  5.554360  10.939141    walking-rider    25848   
...            ...       ...       ...        ...              ...      ...   
1011744  -6.137165  1.372306  0.412540   6.408025  walking-natural   226022   
1011745  -6.232971  1.078513  0.379576   6.349429  walking-natural   226022   
1011746  -6.571576 -0.069279  0.220634   7.075162  walking-natural   226022   
1011747  -7.136520 -0.382676 -0.134757   7.752004  walking-natural   226022   
1011748  -7.621801  1.075187 -1.435447   8.606736  walking-natural   226022   

         subject  ActivityEncoded  
777585         

In [16]:
print(windowedResults[0])


           0         1         2         3         4         5         6    \
0     8.170723  8.170523  8.274468  8.325768  8.069016  7.647190  7.405567   
1     7.887990  7.870641  7.782686  7.819511  7.951974  8.058081  8.175097   
2     7.801493  7.934548  8.041172  8.024676  7.995268  7.978752  7.908119   
3     7.187426  7.487853  7.831477  7.822467  7.573549  7.497467  7.709625   
4     7.698069  7.736879  7.903928  8.034393  7.941801  7.826244  7.866835   
...        ...       ...       ...       ...       ...       ...       ...   
1458 -8.365492 -8.364253 -8.443871 -8.478546 -8.476689 -8.475877 -8.464105   
1459 -7.926858 -7.971660 -7.972241 -8.052828 -8.264358 -8.412105 -8.493876   
1460 -8.353528 -8.259185 -8.415535 -8.616170 -8.568897 -8.426481 -8.410300   
1461 -8.410908 -8.305326 -8.236751 -8.257477 -8.335742 -8.368111 -8.325089   
1462 -8.374409 -8.296256 -8.193131 -8.172891 -8.162346 -8.088046 -8.012266   

           7         8         9    ...       590       591    

In [271]:
featureScaling(windowedResults[0])

In [ ]:
with open('C:\\Users\\emirc\\Desktop\\AAR\\output', 'a') as f:
    dfAsString = windowedResults[0].to_string(header=False, index=False)
    f.write(dfAsString)